In [ ]:
!pip install torch torchaudio
!pip install funasr
!pip install modelscope

drive_model_path = "/content/drive/MyDrive/modelscope_cache"
asr_model = "speech_seaco_paraformer_large_asr_nat-zh-cn-16k-common-vocab8404-pytorch" #SeACoParaformer-zh
vad_model = "speech_fsmn_vad_zh-cn-16k-common-pytorch" #fsmn-vad
punc_model = "punc_ct-transformer_cn-en-common-vocab471067-large" #ct-punc

!modelscope download --model iic/{asr_model} --local_dir {drive_model_path}/iic/{asr_model}
!modelscope download --model iic/{vad_model} --local_dir {drive_model_path}/iic/{vad_model}
!modelscope download --model iic/{punc_model} --local_dir {drive_model_path}/iic/{punc_model}

import yaml
import subprocess
import os

%env TZ=Asia/Shanghai

#获取server_id
try:
    with open('/content/drive/MyDrive/github/bilibili2txt/config.yaml', 'r', encoding='utf-8') as f:
        config_data = yaml.safe_load(f) or {}
    server_id = config_data.get('server', {}).get('server_id', 'digiplanets1')
except Exception:
    server_id = 'digiplanets1'

#配置github
!git config --global user.name {server_id}
!git config --global user.email {server_id}@protonmail.com

#配置证书
!mkdir -p ~/.ssh
!cp /content/drive/MyDrive/digiplanets1_id_ed25519_github ~/.ssh/id_ed25519
!chmod 600 ~/.ssh/id_ed25519
!echo "Host github.com" > ~/.ssh/config
!echo "  HostName github.com" >> ~/.ssh/config
!echo "  User digiplanets1" >> ~/.ssh/config
!echo "  IdentityFile ~/.ssh/id_ed25519" >> ~/.ssh/config
!echo "  StrictHostKeyChecking accept-new" >> ~/.ssh/config

#whisper改成可运行
!chmod +x /content/drive/MyDrive/Faster-Whisper-XXL/faster-whisper-xxl

#环境变量
PROJECT_DIR = '/content/drive/MyDrive/github/bilibili2txt'
CONFIG_PATH = f'{PROJECT_DIR}/config.yaml'

#安装软件
!apt-get update -y
!apt-get install -y ffmpeg git

#升级，并且，安装python库
%cd {PROJECT_DIR}
!git pull
!python -m pip install -U pip
!python -m pip install -r requirements.txt

#清空queue
# 1. 加载配置
with open(CONFIG_PATH, 'r', encoding='utf-8') as f:
    config_data = yaml.safe_load(f) or {}
# 2. 读取字段
queue_repo_dir = config_data.get('queue', {}).get('repo_dir', 'queue')
# 3. 将相对路径自动转换为绝对路径
if not os.path.isabs(queue_repo_dir):
    queue_repo_dir = os.path.abspath(os.path.join(PROJECT_DIR, queue_repo_dir))
print("解析后的 queue 绝对路径:", queue_repo_dir)
!rm -rf {queue_repo_dir}
!git clone https://github.com/您的用户名/您的队列仓库.git {queue_repo_dir}

#运行
!python b2t.py --config {CONFIG_PATH} server run